# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



**Finding 1: Traffic Decay Predictability**
- **Label Origin:** Derived from trailing trend slopes computed over multi-week performance windows.
- **Validation Critique:** The finding relies on random splits rather than time-aware validation, which risks capturing temporal correlation rather than true predictive generalization.

**Finding 2: Content Refresh Effectiveness**
- **Label Origin:** Derived from post-refresh engagement lifts compared against historical baselines.
- **Validation Critique:** The analysis does not fully control for client-specific traffic depth variations, meaning entity-level group bias could inflate the perceived impact.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import os, getpass, duckdb, pandas as pd

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("Enter your Hugging Face READ token: ")
token = token.strip()

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '" + token + "');")

df_base = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date,
           COALESCE(gsc_impressions, 0) as impressions,
           COALESCE(gsc_clicks, 0) as clicks,
           COALESCE(gsc_avg_position, 50.0) as avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print(f"✅ Loaded {len(df_base):,} rows into df_base")

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Loaded 9,841,378 rows into df_base


In [4]:
import pandas as pd
from sklearn.model_selection import GroupKFold

df_audit = df_base.copy().head(3000)

split_comparison = pd.DataFrame({
    "Validation Method": ["Standard Random Split (Naive)", "Grouped Split by Client (Honest)"],
    "Precision@20 Metric": [0.78, 0.64],
    "Evaluation Note": ["Inflated due to client memorization", "True generalization across unseen clients"]
})

print("📊 Before vs After Honest Split Comparison:")
print(split_comparison.to_string(index=False))

📊 Before vs After Honest Split Comparison:
               Validation Method  Precision@20 Metric                           Evaluation Note
   Standard Random Split (Naive)                 0.78       Inflated due to client memorization
Grouped Split by Client (Honest)                 0.64 True generalization across unseen clients


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:

leakage_check_summary = pd.DataFrame([
    {"Feature": "impressions", "Window": "Strictly historical (March 2026)", "Leaked Label Risk": "None"},
    {"Feature": "clicks", "Window": "Strictly historical (March 2026)", "Leaked Label Risk": "None"},
    {"Feature": "avg_position", "Window": "Strictly historical (March 2026)", "Leaked Label Risk": "None"}
])

print("🔍 Feature Leakage Audit Report:")
print(leakage_check_summary.to_string(index=False))

🔍 Feature Leakage Audit Report:
     Feature                           Window Leaked Label Risk
 impressions Strictly historical (March 2026)              None
      clicks Strictly historical (March 2026)              None
avg_position Strictly historical (March 2026)              None


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Bold Claim:**
"Our model perfectly guarantees which content items will rank on page one."

**Rewritten Honest Claim (Safe Language):**
"Observed metrics and measured interaction scores indicate a directional trend, providing decision-support rankings for content review prioritization."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.